# Cell Capacity Calculation

This notebook demonstrates how to calculate cell capacity from a cell design manifest.

## Overview
The `get_cell_capacity` function provides:
- Direct lookup of nominal capacity from KPIs
- First-principles calculation from electrode design parameters
- N/P ratio and limiting electrode identification

In [1]:
import json
from pathlib import Path
from model_library import get_cell_capacity

## 1. Load Cell Manifest

In [2]:
# Load cell manifest
manifest_path = Path("../cells/BYD_Blade_Prismatic_135Ah_manifest.json")

with open(manifest_path, "r") as f:
    cell_design_manifest = json.load(f)
cell_design = cell_design_manifest["cell_design"]
print(f"Cell: {cell_design_manifest['metadata']['name']}")
print(f"Form factor: {cell_design_manifest['cell_design']['form_factor']}")

Cell: BYD Blade Prismatic 135Ah
Form factor: Prismatic


## 2. Calculate Cell Capacity

In [3]:
# Get cell capacity
capacity_result = get_cell_capacity(cell_design)

print("Cell Capacity Results")
print("=" * 50)
print(f"Calculated capacity:             {capacity_result['calculated_capacity_Ah']:.2f} Ah")
print()
print("Electrode Capacities")
print("-" * 50)
print(f"Positive electrode capacity:     {capacity_result['positive_electrode_capacity_Ah']:.2f} Ah")
print(f"Negative electrode capacity:     {capacity_result['negative_electrode_capacity_Ah']:.2f} Ah")
print(f"Limiting electrode:              {capacity_result['limiting_electrode']}")
print(f"N/P ratio:                       {capacity_result['np_ratio']:.2f}")

Cell Capacity Results
Calculated capacity:             135.71 Ah

Electrode Capacities
--------------------------------------------------
Positive electrode capacity:     135.71 Ah
Negative electrode capacity:     172.17 Ah
Limiting electrode:              positive
N/P ratio:                       1.27


## 3. Compare Multiple Cells

In [4]:
# Load and compare multiple cell manifests
cells_dir = Path("../cells")
manifest_files = list(cells_dir.glob("*_manifest.json"))

# Filter out modified/copy files
manifest_files = [f for f in manifest_files if "_mod" not in f.name and "copy" not in f.name]

results = []
for manifest_file in manifest_files:
    with open(manifest_file, "r") as f:
        manifest = json.load(f)
    
    capacity = get_cell_capacity(manifest)
    results.append({
        "name": manifest["metadata"]["name"],
        "form_factor": manifest["cell_design"]["form_factor"],
        "calculated_Ah": capacity["calculated_capacity_Ah"],
        "np_ratio": capacity["np_ratio"],
        "limiting": capacity["limiting_electrode"],
    })

# Display as table
print(f"{'Cell Name':<40} {'Form':<12} {'Nominal':<10} {'Calc':<10} {'N/P':<6} {'Limiting'}")
print("=" * 100)
for r in results:
    print(f"{r['name']:<40} {r['form_factor']:<12} {r['calculated_Ah']:<10.2f} {r['np_ratio']:<6.2f} {r['limiting']}")

Cell Name                                Form         Nominal    Calc       N/P    Limiting
Tesla Model3 Prismatic 160Ah             Prismatic    0.00       0.00   positive
BYD Blade Prismatic 135Ah                Prismatic    0.00       0.00   positive
Tesla ModelY Cylindrical 22Ah            Cylindrical  0.00       0.00   positive
VW ID3 Pouch 80Ah                        Pouch        0.00       0.00   positive
